# Scoring Engine Simulation

This notebook simulates the behavior of the scoring engine under different price scenarios using mathematical models. The goal is to analyze how the scoring engine responds to changes in price and to validate the updated scoring formulas.

In [ ]:
# Define constants and parameters
import math
import pandas as pd
import matplotlib.pyplot as plt

# Constants
COST = 5000  # in RSD
BASELINE_PRICE = 8000  # in RSD
DEMAND_INDEX = 70  # Stable demand index

# Target prices for scenarios
scenarios = {
    "A": 6000,
    "B": 7000,
    "C": 8000,
    "D": 9000,
    "E": 10000
}

In [ ]:
# Implement mathematical functions
def compute_price_fit(target, baseline):
    deviation = abs(target - baseline) / baseline
    return 100 * math.exp(-3 * deviation)

def compute_margin_score(cost, target):
    margin = (target - cost) / target
    return math.tanh(margin * 3) * 100

def compute_sell_probability_score(demand, price_fit):
    return 0.5 * demand + 0.5 * price_fit

def calibrate_sell_probability(score):
    return 1 / (1 + math.exp(-0.08 * (score - 50)))

def compute_final_score(sell_prob_score, margin_score):
    return 0.6 * sell_prob_score + 0.4 * margin_score

In [ ]:
# Simulate scenarios
results = []

for scenario, target_price in scenarios.items():
    price_fit = compute_price_fit(target_price, BASELINE_PRICE)
    margin_score = compute_margin_score(COST, target_price)
    sell_prob_score = compute_sell_probability_score(DEMAND_INDEX, price_fit)
    calibrated_sell_prob = calibrate_sell_probability(sell_prob_score)
    final_score = compute_final_score(sell_prob_score, margin_score)

    results.append({
        "Scenario": scenario,
        "Target Price": target_price,
        "PriceFit": round(price_fit, 2),
        "MarginScore": round(margin_score, 2),
        "SellProbability": round(calibrated_sell_prob, 2),
        "FinalScore": round(final_score, 2)
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)
results_df

In [ ]:
# Visualize results
# Display the results table
print(results_df)

# Plot the results
plt.figure(figsize=(10, 6))

# Plot Final Scores
plt.plot(results_df["Scenario"], results_df["FinalScore"], marker="o", label="Final Score")

# Plot PriceFit and MarginScore
plt.plot(results_df["Scenario"], results_df["PriceFit"], marker="o", label="PriceFit")
plt.plot(results_df["Scenario"], results_df["MarginScore"], marker="o", label="MarginScore")

# Plot SellProbability
plt.plot(results_df["Scenario"], results_df["SellProbability"] * 100, marker="o", label="Sell Probability (%)")

plt.title("Scoring Engine Simulation Results")
plt.xlabel("Scenario")
plt.ylabel("Scores")
plt.legend()
plt.grid()
plt.show()